# CASMI26 submission
Ported prvsiyan 4-channel + HistGBM pipeline (`src/casmi` in the repo).

In [ ]:
!mkdir -p /kaggle/working/casmi

In [ ]:
%%writefile /kaggle/working/casmi/__init__.py
"""CASMI 2026 retrieval pipeline (port of prvsiyan analog-propagation baseline)."""


In [ ]:
%%writefile /kaggle/working/casmi/candidates.py
"""Candidate pool = COCONUT (attached, precomputed fps) U train structures (fingerprinted here), InChIKey14-deduped,
sorted by exact mass. Optional ChEBI+LIPID MAPS (CFG.USE_BIO_DB, off by default as in the notebook).

Cache: data/cache/pool_train_fp.npy / pool_train_meta.parquet (train-structure half, ~5 min once).
"""
from __future__ import annotations

import multiprocessing as mp
import pickle
import time

import numpy as np
import polars as pl

from . import chem, config

C = config.CACHE
_BITS = None


def _init_bits(bits_path):
    global _BITS
    _BITS = np.load(bits_path)


def _fp_and_mass(smi):
    r = chem.raw_fp(smi)
    if r is None:
        return None
    return r[0][_BITS], r[1]


def build_train_fp_cache(cfg: config.CFG, force=False):
    fpp, mp_ = C / 'pool_train_fp.npy', C / 'pool_train_meta.parquet'
    if fpp.exists() and mp_.exists() and not force:
        print('train-structure fingerprint cache exists'); return
    t0 = time.time()
    cm = pickle.load(open(config.COCO_DIR / 'coco_meta.pkl', 'rb'))
    co = [str(k) for k in cm['keys']]
    st = pl.read_parquet(C / 'structs.parquet')
    tr = st.filter(~pl.col('inchikey14').is_in(co))
    smis = tr['normalized_smiles'].to_list()
    print(f'fingerprinting {len(smis):,} training structures not in COCONUT', flush=True)
    ctx = mp.get_context('spawn')
    with ctx.Pool(cfg.WORKERS, initializer=_init_bits, initargs=(str(config.COCO_DIR / 'fp_bits.npy'),)) as p:
        res = p.map(_fp_and_mass, smis, chunksize=500)
    ok = [i for i, r in enumerate(res) if r is not None]
    fp = np.packbits(np.stack([res[i][0] for i in ok]), axis=1)
    mass = np.array([res[i][1] for i in ok])
    np.save(fpp, fp)
    pl.DataFrame(dict(key=tr['inchikey14'].to_numpy()[ok], smiles=np.asarray(smis, dtype=object)[ok].tolist(),
                      mass=mass)).write_parquet(mp_)
    print(f'train fp cache: {len(ok):,} ok / {len(smis):,} ({time.time()-t0:.0f}s)', flush=True)


class Pool:
    """Candidates + packed fingerprints, sorted by exact mass."""

    def __init__(self, cfg: config.CFG):
        t0 = time.time()
        cm = pickle.load(open(config.COCO_DIR / 'coco_meta.pkl', 'rb'))
        fp = [np.load(config.COCO_DIR / 'coco_fp.npy')]
        mass = [np.load(config.COCO_DIR / 'coco_mass.npy')]
        keys = [np.asarray(cm['keys'], dtype=object)]
        smis = [np.asarray(cm['smiles'], dtype=object)]
        src = [np.zeros(len(mass[0]), np.int8)]
        if cfg.USE_BIO_DB:
            bm = pickle.load(open(config.BIO_DIR / 'bio_meta.pkl', 'rb'))
            fp.append(np.load(config.BIO_DIR / 'bio_fp.npy')); mass.append(np.load(config.BIO_DIR / 'bio_mass.npy'))
            keys.append(np.asarray(bm['keys'], dtype=object)); smis.append(np.asarray(bm['smiles'], dtype=object))
            src.append(np.full(len(mass[-1]), 2, np.int8))
        tm = pl.read_parquet(C / 'pool_train_meta.parquet')
        fp.append(np.load(C / 'pool_train_fp.npy')); mass.append(tm['mass'].to_numpy())
        keys.append(tm['key'].to_numpy().astype(object)); smis.append(tm['smiles'].to_numpy().astype(object))
        src.append(np.ones(len(tm), np.int8))
        fp = np.vstack(fp); mass = np.concatenate(mass); keys = np.concatenate(keys)
        smis = np.concatenate(smis); src = np.concatenate(src)
        good = np.isfinite(mass)
        fp, mass, keys, smis, src = fp[good], mass[good], keys[good], smis[good], src[good]
        o = np.argsort(mass)
        self._fp = fp[o]; self.mass = mass[o]; self.keys = keys[o]; self.smiles = smis[o]
        self.src = src[o]  # provenance: NEVER a ranker feature (leaks in the class-2 simulation)
        self.nbits = cm['nbits']
        self.k2i = {k: i for i, k in enumerate(self.keys)}
        print(f'pool: {len(self.mass):,} structures, {self.nbits} bits ({time.time()-t0:.0f}s)', flush=True)

    def window(self, t, ppm):
        a = np.searchsorted(self.mass, t * (1 - ppm / 1e6), 'left')
        b = np.searchsorted(self.mass, t * (1 + ppm / 1e6), 'right')
        return np.arange(a, b)

    def fps(self, idx):
        return np.unpackbits(np.asarray(self._fp[idx]), axis=1)[:, :self.nbits]


In [ ]:
%%writefile /kaggle/working/casmi/chem.py
"""Masses, adducts, neutral-mass derivation and RDKit fingerprints (identical constants to the notebook)."""
from __future__ import annotations

import re

import numpy as np

MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,
            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,
            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)
E = 0.00054857990
PROTON = MASS['H'] - E
H2O = 2 * MASS['H'] + MASS['O']
NH4 = MASS['N'] + 4 * MASS['H']
FORMATE = MASS['C'] + 2 * MASS['H'] + 2 * MASS['O']
ACETATE = 2 * MASS['C'] + 4 * MASS['H'] + 2 * MASS['O']
ADDUCTS = {
    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS['Na'] - E),
    "[M+K]+": (1, 1, MASS['K'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O),
    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),
    "[M+CH3OH+H]+": (1, 1, PROTON + MASS['C'] + 4 * MASS['H'] + MASS['O']),
    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS['C'] + 3 * MASS['H'] + MASS['N']),
    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),
    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS['Cl'] + E), "[M]-": (1, 1, E),
    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS['Na'] - 2 * PROTON),
    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS['Na'] - E), "[2M+NH4]+": (2, 1, NH4 - E),
    "[2M+K]+": (2, 1, MASS['K'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),
    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS['Na'] - 2 * PROTON),
    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),
}
MONO = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, S=31.97207100,
            P=30.97376163, Cl=34.96885268, Br=78.9183371, F=18.99840322, I=126.904473,
            Si=27.9769265325, Se=79.9165213, Na=22.9897692809, K=38.96370668, B=11.0093054,
            As=74.9215965, Fe=55.9349375, D=2.0141017778)
_TOK = re.compile(r'([A-Z][a-z]?)(\d*)')


def formula_mass(f):
    """Neutral monoisotopic mass by arithmetic on a molecular formula (NaN if unknown element)."""
    if not isinstance(f, str) or not f:
        return np.nan
    m = 0.0
    for el, n in _TOK.findall(f):
        if not el:
            continue
        if el not in MONO:
            return np.nan
        m += MONO[el] * (int(n) if n else 1)
    return m


def neutral_mass(mz, adduct):
    mz = np.asarray(mz, np.float64)
    out = np.full(len(mz), np.nan)
    ad = np.asarray(adduct, dtype=object)
    for a, (n, z, d) in ADDUCTS.items():
        m = (ad == a)
        if m.any():
            out[m] = (mz[m] * z - d) / n
    return out


# ---------------------------------------------------------------- fingerprints (ECFP4|ECFP6|RDKit|MACCS)[BITS]
_g: dict = {}


def _fp_init():
    from rdkit import RDLogger
    from rdkit.Chem import rdFingerprintGenerator
    RDLogger.DisableLog('rdApp.*')
    _g['m2'] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g['m3'] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g['rk'] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)


def raw_fp(smi):
    """Full 10,407-bit fingerprint (uint8) and exact mass, or None."""
    from rdkit import Chem
    from rdkit.Chem import MACCSkeys
    from rdkit.Chem.Descriptors import ExactMolWt
    if not _g:
        _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    try:
        fp = np.concatenate([_g['m2'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['m3'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['rk'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])
        return fp, float(ExactMolWt(m))
    except Exception:
        return None


In [ ]:
%%writefile /kaggle/working/casmi/cli.py
"""CLI:  uv run casmi build | cv | predict

  casmi build                               # one-time caches (library, metric keys, train-structure fingerprints)
  casmi cv --n-np 200 --n-e180 300 --tag exp000
  casmi predict --ranker rank_train --out outputs/submission.csv
  casmi predict --ranker cv --cv-tag exp000 --out outputs/submission_cvranker.csv
"""
from __future__ import annotations

import argparse
import glob
import json
import time

from . import config


def _engine(cfg, prefer_instrument='timsTOF'):
    from .candidates import Pool
    from .engine import Engine
    from .frag import Fragmenter
    from .library import Library
    bank = None
    if cfg.USE_FP_MODEL:
        from .fpnet import ModelBank
        paths = sorted(glob.glob(str(config.FP_MODEL_DIR / 'fp_*.pt')))
        bank = ModelBank(paths, device=cfg.DEVICE) if paths else None
    L = Library(cfg)
    P = Pool(cfg)
    fr = Fragmenter(cfg.WORKERS) if cfg.USE_FRAG else None
    return Engine(cfg, L, P, bank, fr, prefer_instrument)


def main(argv=None):
    ap = argparse.ArgumentParser('casmi')
    sub = ap.add_subparsers(dest='cmd', required=True)
    sub.add_parser('build')
    c = sub.add_parser('cv')
    c.add_argument('--n-np', type=int, default=200); c.add_argument('--n-e180', type=int, default=300)
    c.add_argument('--seed', type=int, default=0); c.add_argument('--tag', default='exp000')
    c.add_argument('--no-reuse', action='store_true')
    p = sub.add_parser('predict')
    p.add_argument('--ranker', choices=['rank_train', 'cv'], default='rank_train')
    p.add_argument('--cv-tag', default='exp000')
    p.add_argument('--out', default=str(config.OUTPUTS / 'submission.csv'))
    for s in (c, p):
        s.add_argument('--device', default='cpu'); s.add_argument('--no-fp', action='store_true')
        s.add_argument('--no-frag', action='store_true'); s.add_argument('--workers', type=int, default=None)
    a = ap.parse_args(argv)
    cfg = config.CFG()
    if getattr(a, 'workers', None): cfg.WORKERS = a.workers
    if getattr(a, 'device', None): cfg.DEVICE = a.device
    if getattr(a, 'no_fp', False): cfg.USE_FP_MODEL = False
    if getattr(a, 'no_frag', False): cfg.USE_FRAG = False
    T0 = time.time()
    if a.cmd == 'build':
        from .candidates import build_train_fp_cache
        from .library import build_library_cache
        build_library_cache(cfg); build_train_fp_cache(cfg)
    elif a.cmd == 'cv':
        from .cv import run_cv
        E = _engine(cfg)
        print(f'engine ready {time.time()-T0:.0f}s', flush=True)
        run_cv(E, cfg, a.n_np, a.n_e180, a.seed, a.tag, reuse=not a.no_reuse)
    elif a.cmd == 'predict':
        from . import submit
        from .ranker import Ranker
        queries, pref = submit.test_queries()
        E = _engine(cfg, pref)
        t_eng = time.time() - T0
        if a.ranker == 'rank_train':
            rk = Ranker.from_rank_train(cfg)
        else:
            from .cv import FP_COLS, cv_ranker
            rk = cv_ranker(cfg, a.cv_tag, FP_COLS if not cfg.USE_FP_MODEL else None)
        t1 = time.time()
        rows, diag = submit.predict_test(E, rk, queries, cfg)
        t_pred = time.time() - t1
        df = submit.write_submission(rows, a.out)
        meta = dict(out=a.out, ranker=a.ranker, secs_total=round(time.time() - T0), secs_setup=round(t_eng),
                    secs_predict=round(t_pred), n=len(df), mean_len=float(df['smiles'].str.split(';').list.len().mean()))
        json.dump(meta, open(a.out.replace('.csv', '.meta.json'), 'w'), indent=1)
        print(f'wrote {a.out} {df.shape}; {meta}', flush=True)
    if getattr(locals().get('E', None), 'frag', None) is not None:
        E.frag.close()


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/casmi/config.py
"""Configuration. Defaults reproduce prvsiyan/analog-propagation-casmi-2026-baseline (the 0.335/0.328 config).

Every constant carries the notebook's own measurement comment in research/04-pipeline-analysis.md.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
DATA = Path(os.environ.get("CASMI_DATA", ROOT / "data"))
EXT = Path(os.environ.get("CASMI_EXT", DATA / "ext"))
CACHE = Path(os.environ.get("CASMI_CACHE", DATA / "cache"))
OUTPUTS = Path(os.environ.get("CASMI_OUTPUTS", ROOT / "outputs"))

TRAIN = DATA / "train.parquet"
TEST = DATA / "test.parquet"
SAMPLE = DATA / "sample_submission.csv"

COCO_DIR = EXT / "coconut-casmi26-candidates"
BIO_DIR = EXT / "chebi-lipidmaps-casmi26"
FP_MODEL_DIR = EXT / "casmi26-fp-models-v2"
RANK_TRAIN = EXT / "casmi26-ranker-features" / "rank_train.npz"


@dataclass
class CFG:
    # candidate generation
    PPM_WIN: float = 10.0
    PPM_FALLBACK: float = 30.0
    # spectrum cleaning (library + query side of the similarity channels)
    INT_FLOOR: float = 0.002
    MAX_PEAKS: int = 256
    MZ_TOL: float = 0.01
    INT_POWER: float = 1.0
    ENT_WEIGHT: bool = True
    # analog propagation
    ANALOG_WIN: float = 200.0
    N_ANALOG: int = 200
    SIM_POWER: float = 3.0  # used inside rank_features (P_SIM)
    # ranker
    W1_PRIORS: tuple = (0.30, 0.60)
    SEEDS: tuple = (0, 1, 2, 3)
    GBM: dict = field(default_factory=lambda: dict(max_depth=6, max_iter=500, learning_rate=0.03,
                                                   min_samples_leaf=80, l2_regularization=1.0))
    USE_BIO_DB: bool = False
    CAND_CAP: int = 500
    LIB_MASS_FORMULA: bool = True
    ANALOG_MATCH_INSTR: bool = True
    TOPN: int = 25
    # channels on/off (for ablations)
    USE_FP_MODEL: bool = True
    USE_FRAG: bool = True
    # runtime
    WORKERS: int = max(1, (os.cpu_count() or 4) - 1)
    DEVICE: str = "cpu"  # 'cpu' | 'mps' | 'cuda' for FPNet


In [ ]:
%%writefile /kaggle/working/casmi/cv.py
"""Leak-free CV on timsTOF held-out molecules.

Held-out molecules = structures sampled from enveda-np-examples and enveda-180 (the only timsTOF libraries),
deduplicated by the METRIC key (tautomer-canonical InChIKey14). Each is queried with k of its own timsTOF spectra
(k drawn from the test's spectra-per-molecule distribution). The "target group" G = every library structure
(inchikey14) that shares the target's metric key, so tautomer/stereo duplicates cannot leak.

  split A (class-1-like): hide G's spectra in the query's source library only; other libraries keep theirs.
                          Only molecules whose G has spectra in another library are eligible.
  split B (class-2-like): hide ALL spectra of G (library search + analog representatives); G stays in the pool.
  split C (class-3-like): as B, and also remove every pool member with the target's metric key.

The ranker is retrained inside CV (5 folds by molecule; rows from A (M=0) + B (M=1) of the training folds).
rank_train.npz is NOT used for CV numbers: it stores no molecule identities (only 819 group ids) and was built from
a 250-NP + 569-obscure holdout, i.e. almost certainly contains enveda-np-examples. It is reported only as a
"leaky reference" row.

Known remaining leak: the attached FPNet weights (casmi26-fp-models-v2) were trained on train.parquet with an
unknown structure split, so held-out structures may have been seen by the fingerprint model. cv reports a no-FP
variant and the FP-alone MRR per source library so the size of that leak is visible.
"""
from __future__ import annotations

import json
import pickle
import time
from pathlib import Path

import numpy as np
import polars as pl

from . import config
from .engine import Mask, Query
from .features import FEATURE_NAMES
from .metric import score_key
from .ranker import Ranker

NP_LIB, E180_LIB = 'enveda-np-examples', 'enveda-180'
WEIGHTS = dict(A=0.16, B=0.45, C=0.39)
FP_COLS = list(range(15, 21)) + list(range(25, 31))


# --------------------------------------------------------------------------------------------- holdout
def select_holdout(L, n_np=200, n_e180=300, seed=0):
    rng = np.random.default_rng(seed)
    te = pl.read_parquet(config.TEST, columns=['molecule_id'])
    k_dist = te.group_by('molecule_id').len()['len'].to_numpy()
    chosen, used_mkeys = [], set()
    for lib, n in ((NP_LIB, n_np), (E180_LIB, n_e180)):
        sids = np.unique(L.sid[L.lib == lib])
        rng.shuffle(sids)
        got = 0
        for s in sids:
            mk = L.s_mkey[s]
            if mk is None or mk in used_mkeys:
                continue
            G = np.array(sorted(L.mkey2sids[mk]), np.int64)
            sp = np.concatenate([L.spectra_of(g) for g in G])
            q_sp = sp[(L.lib[sp] == lib) & np.isfinite(L.nm[sp])]
            if len(q_sp) == 0:
                continue
            k = int(min(len(q_sp), rng.choice(k_dist)))
            q_sp = np.sort(rng.choice(q_sp, size=k, replace=False))
            eligible_A = bool((L.lib[sp] != lib).any())
            used_mkeys.add(mk)
            chosen.append(dict(sid=int(s), mkey=mk, smiles=L.s_smiles[s], src_lib=lib, G=G.tolist(),
                               q_rows=q_sp.tolist(), eligible_A=eligible_A))
            got += 1
            if got >= n:
                break
    return chosen


def load_query_rows(rows):
    """Raw (uncleaned) spectra for library row indices, read from train.parquet (row index == library index)."""
    rows = sorted(set(rows))
    df = (pl.scan_parquet(config.TRAIN).with_row_index('row')
            .filter(pl.col('row').is_in(rows))
            .select('row', 'ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz', 'adduct', 'instrument_type',
                    'collision_energy_ev', 'ionization_mode').collect())
    out = {}
    for r in df.iter_rows(named=True):
        out[r['row']] = dict(mz=np.asarray(r['ms2_mzs'], np.float64), it=np.asarray(r['ms2_normalized_intensities'], np.float64),
                             prec=float(r['precursor_mz']), adduct=r['adduct'], instrument=r['instrument_type'],
                             ce=r['collision_energy_ev'], mode=1.0 if r['ionization_mode'] == 'positive' else -1.0)
    return out


def make_query(L, h, raw):
    from .chem import neutral_mass
    specs = [raw[i] for i in h['q_rows']]
    nm = neutral_mass(np.array([s['prec'] for s in specs]), np.array([s['adduct'] for s in specs], dtype=object))
    nm = nm[np.isfinite(nm)]
    if not len(nm):
        return None
    return Query(specs, float(np.median(nm)))


# --------------------------------------------------------------------------------------------- features
def _pack(c, truth):
    if c is None:
        return None
    keys = [score_key(s) for s in c['smiles']]
    Y = np.array([k == truth for k in keys], np.float32)
    return dict(X=c['X'], Y=Y, keys=keys, n=len(keys))


def compute_features(E, holdout, raw, log_every=25):
    L = E.L
    out, t0 = [], time.time()
    tim = dict(logits=0.0, lib=0.0, analog=0.0, cand=0.0)
    for i, h in enumerate(holdout):
        q = make_query(L, h, raw)
        rec = dict(h=h, A=None, B=None, C=None)
        if q is not None:
            truth = h['mkey']; G = np.array(h['G'], np.int64)
            ta = time.time(); z = E.logits(q); tim['logits'] += time.time() - ta
            mB = Mask(excl_sids=G)
            ta = time.time(); libB = E.lib_sim(q, mB); tim['lib'] += time.time() - ta
            ta = time.time(); anB = E.analog_sim(q, mB); tim['analog'] += time.time() - ta
            ta = time.time()
            rec['B'] = _pack(E.candidates(q, libB, anB, z), truth)
            rec['C'] = _pack(E.candidates(q, libB, anB, z, pool_excl_mkeys=frozenset([truth])), truth)
            tim['cand'] += time.time() - ta
            if h['eligible_A']:
                mA = Mask(src_sids=G, src_lib=h['src_lib'])
                bad = lambda idx: mA.spectra_bad(L, idx)
                ov = {int(s): L.rep_excluding(int(s), bad) for s in G}
                ta = time.time(); libA = E.lib_sim(q, mA); tim['lib'] += time.time() - ta
                ta = time.time(); anA = E.analog_sim(q, mA, ov); tim['analog'] += time.time() - ta
                ta = time.time(); rec['A'] = _pack(E.candidates(q, libA, anA, z), truth); tim['cand'] += time.time() - ta
            rec['target'] = q.target; rec['n_spec'] = len(q.spectra)
            rec['lib_max_B'] = float(max(libB.values())) if libB else 0.0
        out.append(rec)
        if (i + 1) % log_every == 0 or i + 1 == len(holdout):
            el = time.time() - t0
            print(f'  cv feats {i+1}/{len(holdout)}  {el:.0f}s ({el/(i+1):.2f}s/mol)  '
                  + ' '.join(f'{k}={v:.0f}s' for k, v in tim.items()), flush=True)
    return out, tim


# --------------------------------------------------------------------------------------------- evaluation
def rr_from_scores(pack, score, topn=25):
    """Reciprocal rank after metric-key dedupe + invalid filtering (exactly what submit.select does)."""
    if pack is None:
        return 0.0
    order = np.argsort(-score, kind='stable')
    seen, r = set(), 0
    for i in order:
        k = pack['keys'][i]
        if k is None or k in seen:
            continue
        seen.add(k); r += 1
        if pack['Y'][i] > 0:
            return 1.0 / r
        if r >= topn:
            break
    return 0.0


def _stack(recs, split, cols_zero=None):
    X, Y, M = [], [], []
    for r in recs:
        p = r[split]
        if p is None or p['n'] == 0:
            continue
        x = p['X'].copy()
        if cols_zero:
            x[:, cols_zero] = 0.0
        X.append(x); Y.append(p['Y']); M.append(np.full(p['n'], 0 if split == 'A' else 1))
    if not X:
        return None
    return np.vstack(X), np.concatenate(Y), np.concatenate(M)


def evaluate(recs, cfg, n_folds=5, seed=0, variants=('full', 'noFP'), use_rank_train=True):
    rng = np.random.default_rng(seed)
    fold = rng.permutation(len(recs)) % n_folds
    res = {}
    t0 = time.time()
    for var in variants:
        cz = FP_COLS if var == 'noFP' else None
        rr = {s: np.full(len(recs), np.nan) for s in 'ABC'}
        for f in range(n_folds):
            tr = [recs[i] for i in range(len(recs)) if fold[i] != f]
            parts = [p for p in (_stack(tr, 'A', cz), _stack(tr, 'B', cz)) if p is not None]
            X = np.vstack([p[0] for p in parts]); Y = np.concatenate([p[1] for p in parts])
            M = np.concatenate([p[2] for p in parts])
            rk = Ranker(cfg).fit(X, Y, M)
            for i in np.where(fold == f)[0]:
                for s in 'ABC':
                    p = recs[i][s]
                    if s == 'A' and not recs[i]['h']['eligible_A']:
                        continue
                    if p is None:
                        rr[s][i] = 0.0; continue
                    x = p['X'].copy()
                    if cz: x[:, cz] = 0.0
                    rr[s][i] = rr_from_scores(p, rk.predict(x))
        res[var] = rr
        print(f'  ranker CV [{var}] done ({time.time()-t0:.0f}s)', flush=True)
    if use_rank_train and config.RANK_TRAIN.exists():
        rk = Ranker.from_rank_train(cfg)
        rr = {s: np.full(len(recs), np.nan) for s in 'ABC'}
        for i, r in enumerate(recs):
            for s in 'ABC':
                if s == 'A' and not r['h']['eligible_A']:
                    continue
                rr[s][i] = rr_from_scores(r[s], rk.predict(r[s]['X'])) if r[s] is not None else 0.0
        res['rank_train(leaky)'] = rr
    # single-channel baselines on split B (no ranker => no ranker leak); FP-alone exposes the FPNet leak
    single = {}
    for name, col in (('lib', 0), ('analog_ap', 5), ('fpnet_fz', 17), ('frag', 21)):
        v = np.full(len(recs), np.nan)
        for i, r in enumerate(recs):
            v[i] = rr_from_scores(r['B'], r['B']['X'][:, col]) if r['B'] is not None else 0.0
        single[name] = v
    return res, single, fold


def summarize(recs, res, single):
    libs = np.array([r['h']['src_lib'] for r in recs])
    lines = []

    def m(v, mask=None):
        v = v if mask is None else v[mask]
        v = v[~np.isnan(v)]
        return (float(v.mean()) if len(v) else float('nan')), len(v)

    table = {}
    for var, rr in res.items():
        row = {}
        for s in 'ABC':
            row[s] = m(rr[s])
            for lib in (NP_LIB, E180_LIB):
                row[f'{s}@{lib}'] = m(rr[s], libs == lib)
        row['total'] = sum(WEIGHTS[s] * row[s][0] for s in 'ABC')
        # per-source weighted totals (A taken from the same source when available, else the pooled A)
        for lib in (NP_LIB, E180_LIB):
            a = row[f'A@{lib}'][0] if row[f'A@{lib}'][1] >= 20 else row['A'][0]
            row[f'total@{lib}'] = WEIGHTS['A'] * a + WEIGHTS['B'] * row[f'B@{lib}'][0] + WEIGHTS['C'] * row[f'C@{lib}'][0]
        table[var] = row
        lines.append(f"{var:>18}: A={row['A'][0]:.4f} (n={row['A'][1]})  B={row['B'][0]:.4f} (n={row['B'][1]})  "
                     f"C={row['C'][0]:.4f}  weighted(16/45/39)={row['total']:.4f}  "
                     f"weighted[np]={row['total@' + NP_LIB]:.4f}  weighted[e180]={row['total@' + E180_LIB]:.4f}")
        lines.append(' ' * 20 + '  '.join(f"{s}[{'np' if lib == NP_LIB else 'e180'}]={row[f'{s}@{lib}'][0]:.4f}(n={row[f'{s}@{lib}'][1]})"
                                            for s in 'AB' for lib in (NP_LIB, E180_LIB)))
    lines.append('single-channel MRR on split B (no ranker):')
    for name, v in single.items():
        lines.append(f"   {name:>10}: all={m(v)[0]:.4f}  np={m(v, libs == NP_LIB)[0]:.4f}  e180={m(v, libs == E180_LIB)[0]:.4f}")
    recall = np.mean([r['B'] is not None and r['B']['Y'].any() for r in recs])
    ncand = np.median([r['B']['n'] for r in recs if r['B'] is not None])
    lines.append(f'split-B window recall={recall:.4f}  median candidates={ncand:.0f}  '
                 f"A-eligible={np.mean([r['h']['eligible_A'] for r in recs]):.3f}")
    return table, '\n'.join(lines)


def run_cv(E, cfg, n_np=200, n_e180=300, seed=0, tag='exp000', reuse=True):
    outdir = config.OUTPUTS / 'cv' / tag
    outdir.mkdir(parents=True, exist_ok=True)
    fpath = outdir / 'feats.pkl'
    T = {}
    t0 = time.time()
    if reuse and fpath.exists():
        recs = pickle.load(open(fpath, 'rb')); tim = {}
        print(f'reusing {fpath} ({len(recs)} molecules)')
    else:
        hold = select_holdout(E.L, n_np, n_e180, seed)
        print(f'holdout: {len(hold)} molecules ({sum(h["src_lib"] == NP_LIB for h in hold)} np-examples, '
              f'{sum(h["eligible_A"] for h in hold)} A-eligible)', flush=True)
        raw = load_query_rows([i for h in hold for i in h['q_rows']])
        T['load_queries'] = time.time() - t0
        recs, tim = compute_features(E, hold, raw)
        T['features'] = time.time() - t0 - T['load_queries']
        pickle.dump(recs, open(fpath, 'wb'))
    t1 = time.time()
    res, single, fold = evaluate(recs, cfg)
    T['ranker_cv'] = time.time() - t1
    table, text = summarize(recs, res, single)
    print(text, flush=True)
    json.dump(dict(table=table, timings=T, channel_timings=tim, n=len(recs)), open(outdir / 'summary.json', 'w'),
              indent=1, default=str)
    (outdir / 'summary.txt').write_text(text + '\n' + json.dumps(dict(timings=T, channel_timings=tim), default=str) + '\n')
    return table, text, T


def cv_ranker(cfg, tag='exp000', cols_zero=None):
    """Ranker fitted on ALL CV rows (A as M=0, B as M=1) -- for test submissions with a leak-free-trained ranker."""
    recs = pickle.load(open(config.OUTPUTS / 'cv' / tag / 'feats.pkl', 'rb'))
    parts = [p for p in (_stack(recs, 'A', cols_zero), _stack(recs, 'B', cols_zero)) if p is not None]
    X = np.vstack([p[0] for p in parts]); Y = np.concatenate([p[1] for p in parts]); M = np.concatenate([p[2] for p in parts])
    return Ranker(cfg).fit(X, Y, M)


__all__ = ['run_cv', 'select_holdout', 'evaluate', 'summarize', 'cv_ranker', 'FEATURE_NAMES']


In [ ]:
%%writefile /kaggle/working/casmi/engine.py
"""Per-molecule scoring: 4 evidence channels -> 31 features for every candidate in the +-10 ppm window.

Channels (see research/04-pipeline-analysis.md):
  1. library   : entropy similarity vs library spectra in the same neutral-mass window   (class-1 evidence)
  2. analog    : mass-shifted entropy similarity vs one representative per structure within +-200 Da,
                 propagated to candidates as max_a sim(a)^p * Tanimoto(c, a)            (class-2 evidence)
  3. frag      : MetFrag-lite explained-intensity fraction
  4. fpnet     : spectrum->fingerprint logits z, candidate score f.z

`Mask` lets CV hide evidence per query (library spectra, analog representatives, pool members) without rebuilding
any index -- see cv.py.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from . import config
from .features import N_FEAT, rank_features
from .metric import score_key
from .spectra import clean, search_clean, search_shift_clean


@dataclass
class Mask:
    """Evidence hidden from one query.
    excl_sids   : structures whose spectra are removed from the library side entirely (split B/C)
    src_sids/src_lib : structures whose spectra are removed only from library `src_lib` (split A)
    pool_excl_mkeys  : metric keys removed from the candidate pool (split C)"""
    excl_sids: np.ndarray = field(default_factory=lambda: np.zeros(0, np.int64))
    src_sids: np.ndarray = field(default_factory=lambda: np.zeros(0, np.int64))
    src_lib: str | None = None
    pool_excl_mkeys: frozenset = frozenset()

    def spectra_bad(self, L, idx):
        if len(idx) == 0:
            return np.zeros(0, bool)
        s = L.sid[idx]
        bad = np.isin(s, self.excl_sids) if len(self.excl_sids) else np.zeros(len(idx), bool)
        if len(self.src_sids) and self.src_lib is not None:
            bad |= np.isin(s, self.src_sids) & (L.lib[idx] == self.src_lib)
        return bad

    @property
    def empty(self):
        return not len(self.excl_sids) and not len(self.src_sids) and not self.pool_excl_mkeys


NO_MASK = Mask()


@dataclass
class Query:
    spectra: list          # dict(mz, it, prec, adduct, instrument, ce, mode)
    target: float          # neutral mass (median over spectra)

    @property
    def peaks(self):
        return [(s['mz'], s['it']) for s in self.spectra]

    @property
    def mode(self):
        return float(np.mean([s['mode'] for s in self.spectra]))


class Engine:
    def __init__(self, cfg: config.CFG, L, pool, bank=None, frag=None, prefer_instrument='timsTOF'):
        self.cfg, self.L, self.pool, self.bank, self.frag = cfg, L, pool, bank, frag
        L.build_rep(prefer_instrument if cfg.ANALOG_MATCH_INSTR else None)
        self.key2sid = {k: i for i, k in enumerate(L.s_key)}
        self.sid2pidx = np.array([pool.k2i.get(k, -1) for k in L.s_key], np.int64)
        self.pidx2sid = np.full(len(pool.keys), -1, np.int64)
        ok = self.sid2pidx >= 0
        self.pidx2sid[self.sid2pidx[ok]] = np.where(ok)[0]

    # ------------------------------------------------------------------ channel 1
    def lib_sim(self, q: Query, mask: Mask = NO_MASK):
        """{sid: max entropy similarity} over library spectra within +-PPM_WIN of the target."""
        cfg, L = self.cfg, self.L
        cand = L.window(q.target, q.target * cfg.PPM_WIN / 1e6)
        if len(cand) and not mask.empty:
            cand = cand[~mask.spectra_bad(L, cand)]
        if len(cand) == 0:
            return {}
        best = np.full(len(cand), -1.0, np.float32)
        for mz, it in q.peaks:
            qm, qp = clean(mz, it, cfg)
            if len(qm) == 0:
                continue
            best = np.maximum(best, search_clean(qm, qp, cand, L.off, L.mz, L.it, cfg.MZ_TOL))
        agg = {}
        for s, v in zip(L.sid[cand].tolist(), best.tolist()):
            if v >= 0 and v > agg.get(s, -1.0):
                agg[s] = v
        return agg

    # ------------------------------------------------------------------ channel 2
    def analog_sim(self, q: Query, mask: Mask = NO_MASK, rep_override=None):
        """[(sid, sim)] top N_ANALOG by mass-shifted entropy similarity over +-ANALOG_WIN Da."""
        cfg, L = self.cfg, self.L
        lo = np.searchsorted(L.rep_nm, q.target - cfg.ANALOG_WIN, 'left')
        hi = np.searchsorted(L.rep_nm, q.target + cfg.ANALOG_WIN, 'right')
        cand = L.rep[lo:hi].copy(); csid = L.rep_sid[lo:hi]; cnm = L.rep_nm[lo:hi].copy()
        if len(cand) == 0:
            return []
        keep = np.ones(len(cand), bool)
        if len(mask.excl_sids):
            keep &= ~np.isin(csid, mask.excl_sids)
        if rep_override:
            pos = {s: i for i, s in enumerate(csid.tolist()) if s in rep_override}
            for s, r in rep_override.items():
                if s in pos:
                    if r < 0:
                        keep[pos[s]] = False
                    else:
                        cand[pos[s]] = r; cnm[pos[s]] = L.nm[r]
        cand, csid, cnm = cand[keep], csid[keep], cnm[keep]
        if len(cand) == 0:
            return []
        shift = (q.target - cnm).astype(np.float32)
        best = np.full(len(cand), -1.0, np.float32)
        any_q = False
        for mz, it in q.peaks:
            qm, qp = clean(mz, it, cfg)
            if len(qm) == 0:
                continue
            any_q = True
            best = np.maximum(best, search_shift_clean(qm, qp, cand, L.off, L.mz, L.it, cfg.MZ_TOL, shift))
        if not any_q:
            return []
        # one rep per sid -> agg == best; library spectra with no peaks keep 0 (as in the notebook)
        best = np.maximum(best, 0.0)
        o = np.argsort(-best, kind='stable')[:cfg.N_ANALOG]
        return list(zip(csid[o].tolist(), best[o].tolist()))

    # ------------------------------------------------------------------ channel 4
    def logits(self, q: Query):
        if self.bank is None or not self.cfg.USE_FP_MODEL:
            return None
        return self.bank.logits(q.spectra)

    # ------------------------------------------------------------------ candidates + features
    def candidates(self, q: Query, lib_hits, analogs, zlog, pool_excl_mkeys=frozenset()):
        cfg, pool = self.cfg, self.pool
        cand = pool.window(q.target, cfg.PPM_WIN)
        if len(cand) == 0:
            cand = pool.window(q.target, cfg.PPM_FALLBACK)
        if len(cand) and pool_excl_mkeys:
            cand = cand[np.array([score_key(pool.smiles[c]) not in pool_excl_mkeys for c in cand], bool)]
        if len(cand) == 0:
            return None
        lv = np.array([lib_hits.get(self.pidx2sid[c], 0.0) if self.pidx2sid[c] >= 0 else 0.0 for c in cand], np.float32)
        if cfg.CAND_CAP and len(cand) > cfg.CAND_CAP:
            coarse = lv * 100.0
            if zlog is not None:
                mz = pool.fps(cand).astype(np.float32) @ zlog
                coarse = coarse + (mz - mz.mean()) / max(float(mz.std()), 1e-9)
            else:
                coarse = coarse - np.abs(pool.mass[cand] - q.target)
            keep = np.argsort(-coarse)[:cfg.CAND_CAP]
            cand, lv = cand[keep], lv[keep]
        cfp = pool.fps(cand)
        csmi = [pool.smiles[c] for c in cand]
        ids, sims = [], []
        for s, v in analogs:
            i = self.sid2pidx[s]
            if i >= 0:
                ids.append(i); sims.append(v)
        afp = pool.fps(np.array(ids)) if ids else None
        fsc = self.frag.scores(csmi, q.peaks, q.mode, cfg) if (self.frag is not None and cfg.USE_FRAG) else None
        X = rank_features(cfp, lv, afp, np.array(sims, np.float32), zlog, fsc, p_sim=cfg.SIM_POWER)[:, :N_FEAT]
        return dict(pidx=cand, smiles=csmi, X=X, lv=lv, top_analog=float(sims[0]) if sims else 0.0)


In [ ]:
%%writefile /kaggle/working/casmi/features.py
"""31 ranker features per candidate (verbatim port of the notebook's rank_features).

Deliberately EXCLUDES pool provenance (src / np_likeness): in the class-2 simulation the answer is always a
training-library structure, so those columns leak.
"""
from __future__ import annotations

import numpy as np

N_FEAT = 31
FEATURE_NAMES = [
    'lv', 'lv_rank', 'lvmax', 'lv_minus_max', 'lv_pos',
    'ap', 'ap_rank', 'apmax', 'ap_minus_max',
    'a1', 'best_tan', 'top_tan', 'mean_tan', 'top_sim', 'log_nc',
    'fz_z', 'fz_rank', 'fz_minus_max', 'fzn_z', 'fzn_rank', 'fz_is_max',
    'fr', 'fr_rank', 'fr_minus_max', 'fr_z',
    'lv_x_mr', 'ap_x_mr', 'agree_lib', 'agree_analog', 'agree_x_lvmax', 'corr_lv_model',
]


def _rank_norm(x):
    o = np.argsort(-x); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)


def _z(x):
    s = x.std()
    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)


def rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None, p_sim=3.0):
    nc = cand_fp.shape[0]
    cf = cand_fp.astype(np.float32); cs = cf.sum(1)
    lv = np.asarray(cand_lib, np.float32)
    lvmax = float(lv.max()) if nc else 0.0
    if analog_fp is not None and len(analog_sim):
        af = analog_fp.astype(np.float32); asum = af.sum(1)
        inter = cf @ af.T
        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)
        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)
        ap = (tan * (w ** p_sim)[None, :]).max(1)
        a1 = (tan * w[None, :]).max(1)
        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])
        mean_tan = (tan * (w ** p_sim)[None, :]).sum(1) / ((w ** p_sim).sum() + 1e-9)
    else:
        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0
    apmax = float(ap.max()) if nc else 0.0
    if model_logits is not None:
        raw = cf @ np.asarray(model_logits, np.float32)
        nrm = raw / np.sqrt(np.maximum(cs, 1.0))
        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),
                 (raw == raw.max()).astype(np.float32)]
    else:
        mfeat = [np.zeros(nc, np.float32)] * 6
    if model_logits is not None and nc:
        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))
        lbest = int(np.argmax(lv)) if lvmax > 0 else -1
        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0
        abest = int(np.argmax(ap)) if apmax > 0 else -1
        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0
        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),
                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]
    else:
        xfeat = [np.zeros(nc, np.float32)] * 6
    if frag is not None:
        fr = np.asarray(frag, np.float32)
        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]
    else:
        ffeat = [np.zeros(nc, np.float32)] * 4
    return np.column_stack([
        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),
        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,
        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),
        np.full(nc, np.log(max(nc, 1))),
        *mfeat, *ffeat, *xfeat,
    ]).astype(np.float32)


In [ ]:
%%writefile /kaggle/working/casmi/fpnet.py
"""Spectrum -> 6,930-bit fingerprint Transformer (CSI:FingerID-style), ranked by f.z. Verbatim port of the notebook."""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",
               "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",
               "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",
               "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>"]
ADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF", "Orbitrap", "QTOF", "IT", "other"]


def instr_family(s):
    if s is None:
        return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4


def prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    """Filter -> window-diversified top-N -> sort by m/z. Returns (mz, sqrt-intensity)."""
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = it >= floor * mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        order = np.argsort(-it)
        bucket = (mz // win).astype(np.int64)
        cnt = {}; sel = []
        for i in order:
            b = bucket[i]; c = cnt.get(b, 0)
            if c < per_win: cnt[b] = c + 1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            ss = set(sel.tolist())
            rest = np.array([i for i in order if i not in ss])
            need = max_peaks - len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it / it.max())
    return mz.astype(np.float32), v.astype(np.float32)


class SinEmb(nn.Module):
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim // 2
        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)
        self.register_buffer('inv', (2 * math.pi) / wav)

    def forward(self, x):
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)


class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h = h
        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(drop)

    def forward(self, x, pad):
        B, N, D = x.shape; y = self.n1(x)
        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        m = (~pad)[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))
        return x + self.drop(self.ff(self.n2(x)))


class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d = d
        self.mz_emb = SinEmb(d); self.nl_emb = SinEmb(d)
        self.pk = nn.Linear(2 * d + 1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d + 3, d)
        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))

    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B, N = mz.shape
        nl = (prec[:, None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec), (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:, 0]
        msk = (~pad[:, 1:]).float().unsqueeze(-1)
        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))


class ModelBank:
    """fp_merged_* models get one merged peak list; fp_single_* get each spectrum. Final z = mean of the two views."""

    def __init__(self, paths, device='cpu'):
        self.dev = device
        self.single, self.merged = [], []
        for pth in sorted(paths):
            ck = torch.load(pth, map_location='cpu', weights_only=False)
            net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).to(device).eval()
            net.load_state_dict(ck['model'])
            (self.merged if 'merged' in str(pth).split('/')[-1] else self.single).append(net)
            self.nbits = ck['nbits']
            print(f'  loaded {str(pth).split("/")[-1]}: d={ck["d"]} layers={ck["layers"]} step={ck.get("step")}')
        torch.set_num_threads(max(1, torch.get_num_threads()))

    @torch.no_grad()
    def logits(self, spectra):
        """spectra: list of dict(mz, it, prec, adduct, instrument, ce(list|float|None), mode(+1/-1)).
        Exactly the notebook's model_logits(sub)."""
        out = []
        if self.single:
            za = self._logits_from(spectra, self.single)
            if za is not None: out.append(za)
        if self.merged:
            mz, it = merge_peaks(spectra)
            r0 = spectra[0]
            zb = self._logits_raw([(mz, it)], self.merged, float(np.median([s['prec'] for s in spectra])),
                                  r0['adduct'], r0['instrument'], 25.0, float(np.mean([s['mode'] for s in spectra])))
            if zb is not None: out.append(zb)
        return np.mean(out, axis=0).astype(np.float32) if out else None

    def _run(self, nets, mz, it, pad, prec, ad, ins, ce, mode):
        T = lambda x: torch.as_tensor(x, device=self.dev)
        args = (T(mz), T(it), T(pad), T(prec), T(ad), T(ins), T(ce), T(mode))
        return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)

    def _logits_from(self, spectra, nets):
        P, rows = [], []
        for r in spectra:
            a, b = prep_peaks(r['mz'], r['it'], float(r['prec']))
            if len(a): P.append((a, b)); rows.append(r)
        if not P: return None
        B = len(P); N = max(len(a) for a, _ in P)
        mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)
        for i, (a, b) in enumerate(P):
            mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False

        def ce_of(r):
            v = r.get('ce')
            try:
                return float(np.mean(np.atleast_1d(v))) if v is not None and len(np.atleast_1d(v)) else 25.0
            except Exception:
                return 25.0
        return self._run(nets, mz, it, pad,
                         np.array([float(r['prec']) for r in rows], np.float32),
                         np.array([ADDUCT_IX.get(r['adduct'], ADDUCT_IX['<unk>']) for r in rows]),
                         np.array([instr_family(r['instrument']) for r in rows]),
                         np.array([ce_of(r) for r in rows], np.float32),
                         np.array([float(r['mode']) for r in rows], np.float32))

    def _logits_raw(self, pairs, nets, prec, adduct, instrument, ce, mode):
        P = [prep_peaks(mz, it, prec) for mz, it in pairs]
        P = [(a, b) for a, b in P if len(a)]
        if not P: return None
        B = len(P); N = max(len(a) for a, _ in P)
        mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)
        for i, (a, b) in enumerate(P):
            mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False
        return self._run(nets, mz, it, pad, np.full(B, prec, np.float32),
                         np.full(B, ADDUCT_IX.get(adduct, ADDUCT_IX['<unk>'])), np.full(B, instr_family(instrument)),
                         np.full(B, ce, np.float32), np.full(B, mode, np.float32))


def merge_peaks(spectra):
    """All of a molecule's peaks collapsed into one pseudo-spectrum (near-duplicate m/z merged, stronger kept)."""
    mz = np.concatenate([np.asarray(s['mz'], float) for s in spectra])
    it = np.concatenate([np.asarray(s['it'], float) / max(float(np.asarray(s['it'], float).max()), 1e-9)
                         if len(s['it']) else np.zeros(0) for s in spectra])
    o = np.argsort(mz); mz, it = mz[o], it[o]
    keep = np.ones(len(mz), bool)
    for j in range(1, len(mz)):
        if mz[j] - mz[j - 1] < 0.005:
            if it[j] >= it[j - 1]: keep[j - 1] = False
            else: keep[j] = False
    return mz[keep], it[keep]


In [ ]:
%%writefile /kaggle/working/casmi/frag.py
"""MetFrag-lite: fraction of sqrt-intensity explained by 1- or 2-bond-break fragments (+-2 H). Verbatim port;
fragment masses are cached per SMILES and computed with a persistent worker pool."""
from __future__ import annotations

import multiprocessing as mp

import numpy as np

from .spectra import _clean

AMU = {'C': 12.0, 'H': 1.00782503207, 'N': 14.0030740048, 'O': 15.9949146196, 'P': 30.97376163,
       'S': 31.97207100, 'F': 18.99840322, 'Cl': 34.96885268, 'Br': 78.9183371, 'I': 126.904473,
       'Na': 22.9897692809, 'K': 38.96370668, 'Si': 27.9769265325, 'B': 11.0093054, 'Se': 79.9165213}
H = AMU['H']; PROTON = H - 0.00054857990


def mol_graph(smi):
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog('rdApp.*')
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    n = m.GetNumAtoms()
    w = np.zeros(n)
    for a in m.GetAtoms():
        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H
    if (w == 0).any(): return None
    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]
    return w, bonds, n


def _components(n, bonds, drop):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(bonds):
        if i in drop: continue
        adj[a].append(b); adj[b].append(a)
    seen = np.zeros(n, bool); comps = []
    for s in range(n):
        if seen[s]: continue
        stack = [s]; seen[s] = True; cur = [s]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)
        comps.append(cur)
    return comps


def fragment_masses(smi, max_breaks=2, max_bonds=34):
    g = mol_graph(smi)
    if g is None: return np.zeros(0)
    w, bonds, n = g
    nb = len(bonds)
    if nb == 0 or nb > max_bonds: return np.array([w.sum()])
    out = {w.sum()}
    for i in range(nb):
        for c in _components(n, bonds, {i}):
            out.add(float(w[c].sum()))
    if max_breaks >= 2:
        for i in range(nb):
            for j in range(i + 1, nb):
                for c in _components(n, bonds, {i, j}):
                    out.add(float(w[c].sum()))
    return np.array(sorted(out))


def explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):
    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0
    ion = []
    for dh in h_shifts:
        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))
    ion = np.sort(np.concatenate(ion))
    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()
    if tot <= 0: return 0.0
    idx = np.searchsorted(ion, peak_mz)
    ok = np.zeros(len(peak_mz), bool)
    for off in (-1, 0):
        k = np.clip(idx + off, 0, len(ion) - 1)
        ok |= np.abs(ion[k] - peak_mz) <= tol
    return float(w[ok].sum() / tot)


def _frag_masses(smi):
    try:
        return fragment_masses(smi)
    except Exception:
        return np.zeros(0)


class Fragmenter:
    def __init__(self, workers=4):
        self.cache = {}
        self.workers = workers
        self._pool = None

    def _p(self):
        if self._pool is None and self.workers > 1:
            self._pool = mp.get_context('spawn').Pool(self.workers)
        return self._pool

    def masses(self, smiles):
        need = [s for s in dict.fromkeys(smiles) if s not in self.cache]
        if need:
            p = self._p()
            res = p.map(_frag_masses, need, chunksize=4) if p is not None and len(need) > 8 else list(map(_frag_masses, need))
            self.cache.update(zip(need, res))
        return [self.cache[s] for s in smiles]

    def scores(self, cand_smiles, specs, mode, cfg):
        """Max over the molecule's spectra of the explain-score, per candidate."""
        frags = self.masses(list(cand_smiles))
        peaks = []
        for mz, it in specs:
            m2, i2 = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32), cfg.INT_FLOOR, cfg.MAX_PEAKS, 1.0, False)
            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))
        out = np.zeros(len(cand_smiles), np.float32)
        for j, f in enumerate(frags):
            out[j] = max((explain_score(f, a, b, mode=mode, tol=cfg.MZ_TOL) for a, b in peaks), default=0.0)
        return out

    def close(self):
        if self._pool is not None:
            self._pool.close(); self._pool = None


In [ ]:
%%writefile /kaggle/working/casmi/library.py
"""Reference spectral library (train.parquet) as a compact on-disk cache + mass index.

Cache (data/cache/lib_*):
  lib_off.npy / lib_mz.npy / lib_it.npy   pre-cleaned peaks (CSR, float32) -- see spectra.clean_batch
  lib_meta.parquet                        per spectrum: sid, lib, instrument, adduct, mode, precursor_mz, nm, npk_raw, ce
  structs.parquet                         per structure (unique inchikey14): sid, inchikey14, smiles, formula, mkey
mkey = metric score key (tautomer-canonical InChIKey14) -- used for CV masking/labels, never as a feature.
"""
from __future__ import annotations

import time

import numpy as np
import polars as pl
import pyarrow.parquet as pq

from . import chem, config
from .spectra import clean_batch

C = config.CACHE


def _lib_paths():
    return dict(off=C / 'lib_off.npy', mz=C / 'lib_mz.npy', it=C / 'lib_it.npy',
                meta=C / 'lib_meta.parquet', structs=C / 'structs.parquet')


def build_library_cache(cfg: config.CFG, force=False):
    P = _lib_paths()
    if all(p.exists() for p in P.values()) and not force:
        print('library cache exists'); return
    C.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    # ---- structures
    lf = pl.scan_parquet(config.TRAIN)
    st = (lf.select('inchikey14', 'normalized_smiles', 'molecular_formula').with_row_index('row')
            .group_by('inchikey14').agg(pl.col('row').min(), pl.col('normalized_smiles').first(),
                                        pl.col('molecular_formula').first())
            .sort('row').drop('row').with_row_index('sid').collect())
    print(f'structures: {len(st):,}  ({time.time()-t0:.0f}s)', flush=True)
    from .metric import score_keys_parallel
    mk = score_keys_parallel(st['normalized_smiles'].to_list(), workers=cfg.WORKERS)
    st = st.with_columns(pl.Series('mkey', mk, dtype=pl.String))
    st.write_parquet(P['structs'])
    print(f'metric keys done: {st["mkey"].null_count()} null, {st["mkey"].n_unique():,} unique '
          f'({time.time()-t0:.0f}s)', flush=True)
    k2sid = dict(zip(st['inchikey14'].to_list(), st['sid'].to_list()))
    # ---- spectra, row group by row group
    pf = pq.ParquetFile(config.TRAIN)
    metas, offs, mzs, its = [], [], [], []
    base = 0
    fm_cache = {}
    for rg in range(pf.num_row_groups):
        t = pf.read_row_group(rg, columns=['ingest_lib', 'inchikey14', 'molecular_formula', 'instrument_type', 'adduct',
                                           'ionization_mode', 'precursor_mz', 'ms2_mzs', 'ms2_normalized_intensities',
                                           'collision_energy_ev'])
        mzc = t.column('ms2_mzs').combine_chunks(); itc = t.column('ms2_normalized_intensities').combine_chunks()
        off = mzc.offsets.to_numpy().astype(np.int64)
        off = off - off[0]
        mz = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
        it = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)
        noff, cmz, cit = clean_batch(off, mz, it, cfg.INT_FLOOR, cfg.MAX_PEAKS, cfg.INT_POWER, cfg.ENT_WEIGHT)
        offs.append(noff[1:] + base); base += int(noff[-1]); mzs.append(cmz); its.append(cit)
        df = pl.from_arrow(t.drop(['ms2_mzs', 'ms2_normalized_intensities']))
        fo = df['molecular_formula'].to_list()
        for f in set(fo):
            if f not in fm_cache:
                fm_cache[f] = chem.formula_mass(f)
        nmf = np.array([fm_cache[f] for f in fo], np.float64)
        nmp = chem.neutral_mass(df['precursor_mz'].to_numpy(), df['adduct'].to_numpy())
        ce = df['collision_energy_ev'].list.mean().fill_null(-1.0)
        metas.append(pl.DataFrame(dict(
            sid=np.array([k2sid[k] for k in df['inchikey14'].to_list()], np.int32),
            lib=df['ingest_lib'], instrument=df['instrument_type'], adduct=df['adduct'], mode=df['ionization_mode'],
            precursor_mz=df['precursor_mz'], nm_formula=nmf, nm_prec=nmp, npk_raw=np.diff(off).astype(np.int32),
            ce=ce)))
        print(f'  row group {rg+1}/{pf.num_row_groups}: {len(off)-1:,} spectra, {len(cmz):,} cleaned peaks '
              f'({time.time()-t0:.0f}s)', flush=True)
    np.save(P['off'], np.concatenate([np.zeros(1, np.int64)] + offs))
    np.save(P['mz'], np.concatenate(mzs)); np.save(P['it'], np.concatenate(its))
    pl.concat(metas).write_parquet(P['meta'])
    print(f'library cache written ({time.time()-t0:.0f}s)', flush=True)


class Library:
    def __init__(self, cfg: config.CFG):
        P = _lib_paths()
        t0 = time.time()
        self.cfg = cfg
        self.off = np.load(P['off']); self.mz = np.load(P['mz']); self.it = np.load(P['it'])
        meta = pl.read_parquet(P['meta'])
        self.meta = meta
        self.sid = meta['sid'].to_numpy()
        self.lib = meta['lib'].to_numpy()
        self.instrument = meta['instrument'].to_numpy()
        nmf = meta['nm_formula'].to_numpy(); nmp = meta['nm_prec'].to_numpy()
        self.nm = np.where(np.isfinite(nmf), nmf, nmp) if cfg.LIB_MASS_FORMULA else nmp
        ok = np.isfinite(self.nm)
        self.order = np.argsort(np.where(ok, self.nm, 1e18), kind='mergesort')
        self.n_ok = int(ok.sum())
        self.snm = self.nm[self.order][:self.n_ok]
        self.npk = meta['npk_raw'].to_numpy()
        st = pl.read_parquet(P['structs'])
        self.structs = st
        self.s_key = st['inchikey14'].to_numpy()
        self.s_smiles = st['normalized_smiles'].to_numpy()
        self.s_mkey = st['mkey'].to_numpy()
        # spectra grouped by structure (for per-query masking / representative re-selection)
        o = np.argsort(self.sid, kind='mergesort')
        self._by_sid = o
        self._sid_bounds = np.searchsorted(self.sid[o], np.arange(len(st) + 1))
        # mkey -> sids (tautomer/stereo duplicates share one metric key)
        self.mkey2sids = {}
        for s, k in enumerate(self.s_mkey):
            if k is not None:
                self.mkey2sids.setdefault(k, []).append(s)
        print(f'library: {len(self.sid):,} spectra / {len(st):,} structures, {len(self.mz):,} cleaned peaks '
              f'({time.time()-t0:.0f}s)', flush=True)

    def spectra_of(self, sid):
        return self._by_sid[self._sid_bounds[sid]:self._sid_bounds[sid + 1]]

    def window(self, target, tol):
        lo = np.searchsorted(self.snm, target - tol, 'left')
        hi = np.searchsorted(self.snm, target + tol, 'right')
        return self.order[lo:hi]

    # ------------------------------------------------------------------ analog representatives
    def build_rep(self, prefer_instrument=None):
        """One representative spectrum per structure (prefer the test instrument, then the richest raw spectrum)."""
        pref = None
        if prefer_instrument:
            pref = str(prefer_instrument).lower().replace('-', '').replace(' ', '')
        self._pref = pref
        match = self._instr_match()
        self._match = match
        n = len(self.structs)
        # vectorised: sort by (sid, match desc, npk desc, index asc) and take first of each sid
        idx = np.arange(len(self.sid))
        keys = np.lexsort((idx, -self.npk.astype(np.int64), -match.astype(np.int64), self.sid))
        first = np.ones(len(keys), bool)
        ss = self.sid[keys]
        first[1:] = ss[1:] != ss[:-1]
        rep = np.sort(keys[first])
        nm = self.nm[rep]; ok = np.isfinite(nm)
        rep = rep[ok]; nm = nm[ok]
        o = np.argsort(nm)
        self.rep, self.rep_sid, self.rep_nm = rep[o], self.sid[rep[o]], nm[o]
        self.rep_of_sid = np.full(n, -1, np.int64)
        self.rep_of_sid[self.rep_sid] = self.rep
        print(f'  analog representatives: {len(self.rep):,} (instrument pref {prefer_instrument!r}: '
              f'{match.mean():.1%} of spectra match)', flush=True)

    def _instr_match(self):
        if self._pref is None:
            return np.zeros(len(self.sid), bool)
        u, inv = np.unique(np.asarray([x if isinstance(x, str) else '' for x in self.instrument], dtype=object),
                           return_inverse=True)
        um = np.array([x.lower().replace('-', '').replace(' ', '') == self._pref for x in u])
        return um[inv]

    def rep_excluding(self, sid, bad_spectra_mask_fn):
        """Best representative of `sid` among its spectra that survive a mask (same rule as build_rep)."""
        sp = self.spectra_of(sid)
        sp = sp[~bad_spectra_mask_fn(sp)]
        sp = sp[np.isfinite(self.nm[sp])]
        if len(sp) == 0:
            return -1
        o = np.lexsort((sp, -self.npk[sp].astype(np.int64), -self._match[sp].astype(np.int64)))
        return int(sp[o[0]])


In [ ]:
%%writefile /kaggle/working/casmi/metric.py
"""Official CASMI metric, re-implemented exactly (kaggle.com/code/metric/casmi-mean-reciprocal-rank).

key(smiles) = InChIKey(TautomerEnumerator().Canonicalize(MolFromSmiles(smiles)))[:14]  with RDKit 2026.03.3.
Score per molecule = 1/rank of the first guess whose key is in the accepted set, 0 if none within max_rank.
Empty entries (A;;B) are dropped before ranking; a trailing ';' is ignored.
"""
from __future__ import annotations

import warnings
from functools import lru_cache

import numpy as np

EXPECTED_RDKIT_VERSION = '2026.03.3'
_T = None


def _taut():
    global _T
    if _T is None:
        import rdkit
        from rdkit import RDLogger
        from rdkit.Chem.MolStandardize import rdMolStandardize
        RDLogger.DisableLog('rdApp.*')
        if rdkit.__version__ != EXPECTED_RDKIT_VERSION:
            warnings.warn(f'RDKit {rdkit.__version__} != metric {EXPECTED_RDKIT_VERSION}: tautomer keys may differ')
        _T = rdMolStandardize.TautomerEnumerator()
    return _T


@lru_cache(maxsize=2_000_000)
def score_key(smiles):
    """Tautomer-canonical InChIKey14, or None if RDKit cannot parse/canonicalize (== metric's canonicalize_smiles_to_ik14)."""
    if not isinstance(smiles, str):
        return None
    from rdkit import Chem
    te = _taut()
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        cm = te.Canonicalize(mol)
        ik = Chem.MolToInchiKey(cm)
        return ik[:14] if ik else None
    except (ValueError, RuntimeError, TypeError):
        return None


def score_keys_parallel(smiles_list, workers=8, chunksize=256):
    """Compute score keys for many SMILES with a process pool (order preserved)."""
    import multiprocessing as mp
    if workers <= 1 or len(smiles_list) < 2000:
        return [score_key(s) for s in smiles_list]
    ctx = mp.get_context('spawn')
    with ctx.Pool(workers) as p:
        return p.map(_score_key_plain, smiles_list, chunksize=chunksize)


def _score_key_plain(s):
    return score_key(s)


def parse_guesses(raw):
    return [c.strip() for c in str(raw).split(';') if c.strip()]


def reciprocal_rank(guesses, accepted_keys, max_rank=25):
    for r, g in enumerate(guesses[:max_rank], 1):
        if score_key(g) in accepted_keys:
            return 1.0 / r
    return 0.0


def mrr(solution: dict, submission: dict, max_rank=25):
    """solution: {molecule_id: answer smiles (comma-separated alternatives allowed)};
       submission: {molecule_id: 'smi1;smi2;...' or list}. Missing molecules score 0."""
    rr = []
    for mid, ans in solution.items():
        acc = {k for k in map(score_key, [s for s in ans.split(',') if s]) if k}
        if not acc:
            raise ValueError(f'unparseable solution for {mid}: {ans!r}')
        g = submission.get(mid, [])
        if isinstance(g, str):
            g = parse_guesses(g)
        if len(g) > max_rank:
            raise ValueError(f'{mid}: {len(g)} guesses > {max_rank}')
        rr.append(reciprocal_rank(g, acc, max_rank))
    return float(np.mean(rr)) if rr else 0.0


def rank_of_key(ranked_keys, truth_key, max_rank=25):
    """1-based rank of truth among already-keyed guesses, or 0 if absent (fast path for CV)."""
    for r, k in enumerate(ranked_keys[:max_rank], 1):
        if k == truth_key:
            return r
    return 0


In [ ]:
%%writefile /kaggle/working/casmi/ranker.py
"""Calibrated ranker: HistGradientBoosting P(candidate is the answer), bagged over class priors x seeds.

Training rows come from a class-1 simulation (M=0) and a class-2 simulation (M=1); sample weight w1 on M=0 rows and
1-w1 on M=1 rows, averaged over CFG.W1_PRIORS x CFG.SEEDS (the notebook's (0.30, 0.60) x (0,1,2,3)).
"""
from __future__ import annotations

import time

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier

from . import config


class Ranker:
    def __init__(self, cfg: config.CFG, priors=None, seeds=None):
        self.cfg = cfg
        self.priors = tuple(priors if priors is not None else cfg.W1_PRIORS)
        self.seeds = tuple(seeds if seeds is not None else cfg.SEEDS)
        self.models = []

    def fit(self, X, Y, M):
        t0 = time.time()
        self.models = []
        for w1 in self.priors:
            W = np.where(M == 0, w1, 1.0 - w1)
            for sd in self.seeds:
                m = HistGradientBoostingClassifier(random_state=sd, **self.cfg.GBM)
                m.fit(X, Y, sample_weight=W)
                self.models.append(m)
        self.fit_secs = time.time() - t0
        return self

    def predict(self, X):
        return np.mean([m.predict_proba(X)[:, 1] for m in self.models], axis=0)

    @classmethod
    def from_rank_train(cls, cfg: config.CFG, path=config.RANK_TRAIN):
        """The notebook's shipped simulation rows (819 query groups; identities unknown -> not usable inside CV)."""
        z = np.load(path)
        r = cls(cfg).fit(z['X'], z['Y'], z['M'])
        print(f'ranker (rank_train.npz): {len(r.models)} GBMs on {len(z["Y"]):,} rows ({r.fit_secs:.0f}s)', flush=True)
        return r


In [ ]:
%%writefile /kaggle/working/casmi/spectra.py
"""Similarity kernels: entropy-weighted spectral entropy similarity (Li et al. 2021) + mass-shifted variant.

Verbatim numba port of the notebook kernels. One change for speed that preserves results exactly:
library spectra are cleaned ONCE at cache-build time (clean_batch) instead of on every comparison,
so `search*` here take pre-cleaned library arrays.
"""
from __future__ import annotations

import numpy as np
from numba import njit, prange


@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx:
            mx = it[i]
    if mx <= 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx
    c = 0
    for i in range(n):
        if it[i] >= thr:
            c += 1
    idx = np.empty(c, np.int64)
    j = 0
    for i in range(n):
        if it[i] >= thr:
            idx[j] = i
            j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c):
            v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk):
            k2[i] = idx[o[i]]
        k2.sort()
        idx = k2
        c = topk
    om = np.empty(c, np.float32)
    oi = np.empty(c, np.float32)
    s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]
        v = it[idx[i]] ** power
        oi[i] = v
        s += v
    if s > 0:
        for i in range(c):
            oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0:
                S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S
            s2 = 0.0
            for i in range(c):
                oi[i] = oi[i] ** w
                s2 += oi[i]
            if s2 > 0:
                for i in range(c):
                    oi[i] /= s2
    return om, oi


@njit(cache=True)
def _clean_counts(off, mz, it, floor, topk, power, ent_weight):
    n = len(off) - 1
    cnt = np.zeros(n, np.int64)
    for k in range(n):
        a = off[k]; b = off[k + 1]
        if b > a:
            cm, cp = _clean(mz[a:b], it[a:b], floor, topk, power, ent_weight)
            cnt[k] = len(cm)
    return cnt


@njit(cache=True, parallel=True)
def _clean_fill(off, mz, it, floor, topk, power, ent_weight, noff, omz, oit):
    n = len(off) - 1
    for k in prange(n):
        a = off[k]; b = off[k + 1]
        if b > a:
            cm, cp = _clean(mz[a:b], it[a:b], floor, topk, power, ent_weight)
            o = noff[k]
            for i in range(len(cm)):
                omz[o + i] = cm[i]
                oit[o + i] = cp[i]


def clean_batch(off, mz, it, floor, topk, power, ent_weight):
    """Clean a CSR batch of spectra. Returns (new_off int64, mz float32, it float32)."""
    mz = np.ascontiguousarray(mz, np.float32); it = np.ascontiguousarray(it, np.float32)
    off = np.ascontiguousarray(off, np.int64)
    cnt = _clean_counts_par(off, mz, it, floor, topk, power, ent_weight)
    noff = np.zeros(len(cnt) + 1, np.int64)
    np.cumsum(cnt, out=noff[1:])
    omz = np.empty(noff[-1], np.float32); oit = np.empty(noff[-1], np.float32)
    _clean_fill(off, mz, it, floor, topk, power, ent_weight, noff, omz, oit)
    return noff, omz, oit


@njit(cache=True, parallel=True)
def _clean_counts_par(off, mz, it, floor, topk, power, ent_weight):
    n = len(off) - 1
    cnt = np.zeros(n, np.int64)
    for k in prange(n):
        a = off[k]; b = off[k + 1]
        if b > a:
            cm, cp = _clean(mz[a:b], it[a:b], floor, topk, power, ent_weight)
            cnt[k] = len(cm)
    return cnt


@njit(cache=True, fastmath=True)
def entropy_sim(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0:
            SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0:
            SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        d = qmz[i] - cmz[j]
        if d < -tol:
            buf[b] = qp[i]; i += 1; b += 1
        elif d > tol:
            buf[b] = cp[j]; j += 1; b += 1
        else:
            buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n:
        buf[b] = qp[i]; i += 1; b += 1
    while j < m:
        buf[b] = cp[j]; j += 1; b += 1
    for x in range(b):
        tot += buf[x]
    if tot <= 0:
        return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0:
            SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)


@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):
    a = entropy_sim(qmz, qp, cmz, cp, tol)
    if shift > -0.001 and shift < 0.001:
        return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)):
        sm[i] = cmz[i] + shift
    b = entropy_sim(qmz, qp, sm, cp, tol)
    return a if a > b else b


@njit(cache=True, fastmath=True, parallel=True)
def search_clean(qmz, qp, cand, off, allmz, allin, tol):
    """Entropy similarity of one cleaned query vs pre-cleaned library spectra `cand`."""
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a:
            continue
        out[k] = entropy_sim(qmz, qp, allmz[a:b], allin[a:b], tol)
    return out


@njit(cache=True, fastmath=True, parallel=True)
def search_shift_clean(qmz, qp, cand, off, allmz, allin, tol, shift):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a:
            continue
        out[k] = entropy_sim_shift(qmz, qp, allmz[a:b], allin[a:b], tol, shift[k])
    return out


def clean(mz, it, cfg):
    return _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                  cfg.INT_FLOOR, cfg.MAX_PEAKS, cfg.INT_POWER, cfg.ENT_WEIGHT)


In [ ]:
%%writefile /kaggle/working/casmi/submit.py
"""Final list selection + submission writing/validation.

Rules (one invalid SMILES reportedly zeroes the whole submission on Kaggle):
  * every output SMILES must parse AND tautomer-canonicalize with RDKit 2026.03.3 (score_key not None)
  * dedupe by the metric key (tautomer-canonical InChIKey14) so no slot is wasted on a duplicate
  * <= 25 per molecule, no empty entries, no ';' inside a SMILES, fallback 'CCO' if nothing survives
"""
from __future__ import annotations

import time

import numpy as np
import polars as pl

from . import config
from .engine import Query
from .metric import parse_guesses, score_key

FALLBACK = 'CCO'


def select(smiles_ranked, n=25):
    """Walk a ranked SMILES list, keep valid & metric-unique entries. Returns (smiles, keys)."""
    out, keys, seen = [], [], set()
    for s in smiles_ranked:
        if not isinstance(s, str) or not s or ';' in s or ',' in s or s != s.strip():
            continue
        k = score_key(s)
        if k is None or k in seen:
            continue
        seen.add(k); out.append(s); keys.append(k)
        if len(out) >= n:
            break
    return out, keys


def validate_submission(df: pl.DataFrame, sample: pl.DataFrame, max_rank=25):
    assert df.columns[:2] == ['molecule_id', 'smiles'], df.columns
    assert len(df) == len(sample) and set(df['molecule_id']) == set(sample['molecule_id'])
    assert df['molecule_id'].is_unique().all(), 'duplicate molecule_id'
    assert df['smiles'].null_count() == 0
    n_bad = n_dup = 0
    for s in df['smiles'].to_list():
        g = parse_guesses(s)
        assert 1 <= len(g) <= max_rank, (len(g), s[:80])
        assert len(g) == len(s.split(';')), 'empty entry'
        ks = [score_key(x) for x in g]
        n_bad += sum(k is None for k in ks)
        n_dup += len(ks) - len(set(ks))
    assert n_bad == 0, f'{n_bad} unparseable SMILES'
    assert n_dup == 0, f'{n_dup} metric-duplicate SMILES'
    return True


def test_queries(path=config.TEST):
    from .chem import neutral_mass
    te = pl.read_parquet(path)
    nm = neutral_mass(te['precursor_mz'].to_numpy(), te['adduct'].to_numpy())
    te = te.with_columns(pl.Series('nm', nm))
    out = {}
    for mid, sub in te.group_by('molecule_id', maintain_order=True):
        mid = mid[0]
        nms = sub['nm'].to_numpy(); nms = nms[np.isfinite(nms)]
        if not len(nms):
            out[mid] = None; continue
        specs = [dict(mz=np.asarray(r['ms2_mzs'], np.float64), it=np.asarray(r['ms2_normalized_intensities'], np.float64),
                      prec=float(r['precursor_mz']), adduct=r['adduct'], instrument=r['instrument_type'],
                      ce=r['collision_energy_ev'], mode=1.0 if r['ionization_mode'] == 'positive' else -1.0)
                 for r in sub.iter_rows(named=True)]
        out[mid] = Query(specs, float(np.median(nms)))
    instr = te['instrument_type'].drop_nulls()
    pref = instr.mode().sort()[0] if len(instr) else None
    return out, pref


def predict_test(E, ranker, queries, cfg, log_every=50):
    rows, diag = {}, []
    t0 = time.time()
    for gi, (mid, q) in enumerate(queries.items()):
        smis = []
        if q is not None:
            lib = E.lib_sim(q); an = E.analog_sim(q); z = E.logits(q)
            c = E.candidates(q, lib, an, z)
            if c is not None:
                p = ranker.predict(c['X'])
                order = np.argsort(-p)
                ranked = [c["smiles"][i] for i in order]
                smis, _ = select(ranked, cfg.TOPN)
                diag.append(dict(molecule_id=mid, target=q.target, n_cand=len(c['smiles']), lib_max=float(c['lv'].max()),
                                 top_analog=c['top_analog'], top_p=float(p[order[0]])))
        rows[mid] = smis or [FALLBACK]
        if gi % log_every == 0:
            print(f'  {gi}/{len(queries)}  {time.time()-t0:.0f}s', flush=True)
    return rows, diag


def write_submission(rows: dict, out_path, sample_path=config.SAMPLE):
    sample = pl.read_csv(sample_path)
    df = pl.DataFrame(dict(molecule_id=sample['molecule_id'],
                           smiles=[';'.join(rows.get(m, [FALLBACK])) for m in sample['molecule_id'].to_list()]))
    validate_submission(df, sample)
    df.write_csv(out_path)
    return df


In [ ]:
import glob, os, subprocess, sys
from pathlib import Path

# Offline RDKit pinned to the metric's version.
py = f"cp{sys.version_info.major}{sys.version_info.minor}"
wheel = glob.glob(f"/kaggle/input/**/rdkit-2026.3.3-{py}-*.whl", recursive=True)[0]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-index", wheel], check=True)

# Lay the inputs out the way casmi.config expects: data/{train,test}.parquet and data/ext/<dataset>.
data = Path("/kaggle/working/data"); (data / "ext").mkdir(parents=True, exist_ok=True)
comp = Path(sorted(p for p in glob.glob("/kaggle/input/**/test.parquet", recursive=True) if "enveda-CASMI26-molecule-id-mass-spectra" in p)[0]).parent
for f in ["train.parquet", "test.parquet", "sample_submission.csv"]:
    (data / f).unlink(missing_ok=True); (data / f).symlink_to(comp / f)
for name in ['casmi26-fp-models-v2', 'casmi26-ranker-features', 'chebi-lipidmaps-casmi26', 'coconut-casmi26-candidates']:
    src = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(p)][0]
    (data / "ext" / name).unlink(missing_ok=True); (data / "ext" / name).symlink_to(src)
os.environ["CASMI_DATA"] = str(data)
os.environ["CASMI_OUTPUTS"] = "/kaggle/working/outputs"
sys.path.insert(0, "/kaggle/working")
print(comp, sorted(os.listdir(data / "ext")))


In [ ]:
import time; t0 = time.time()
from casmi.cli import main
main(['build'])
print(f'build {time.time()-t0:.0f}s')

In [ ]:
t0 = time.time()
main(['predict', '--ranker', 'rank_train', '--device', 'cuda' if __import__('torch').cuda.is_available() else 'cpu',
      '--out', '/kaggle/working/submission.csv'])
print(f'predict {time.time()-t0:.0f}s')

In [ ]:
import polars as pl
from rdkit import Chem
sub = pl.read_csv('/kaggle/working/submission.csv')
bad = [s for row in sub['smiles'] for s in row.split(';') if Chem.MolFromSmiles(s) is None]
assert not bad, bad[:5]
print(sub.shape, 'all SMILES parse')